In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install Sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 5.1 MB/s eta 0:00:00


In [ ]:
#Import library yang dibutuhkan
import nltk
import re
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from nltk.tokenize import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemover, ArrayDictionary
import string

In [ ]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
#Membuka file data tweets
data_tweet = pd.read_csv('/content/drive/MyDrive/Int Conference/Merged LPDP 2021-2026.csv', encoding='latin-1')
data_tweet = data_tweet.astype(str)

In [ ]:
#Proses filter
def filtering_text(text):
    # mengubah tweet menjadi huruf kecil
    text = text.lower()
    # menghilangkan url
    text = re.sub(r'https?:\/\/\S+','',text)
    # menghilangkan mention, link, hastag
    text = ' '.join(re.sub(r"([@#][A-Za-z0-9]+)|(\w+:\/\/\S+)"," ", text).split())
    #menghilangkan karakter byte (b')
    text = re.sub(r'(b\'{1,2})',"", text)
    # menghilangkan yang bukan huruf
    text = re.sub('[^a-zA-Z]', ' ', text)
    # menghilangkan digit angka
    text = re.sub(r'\d+', '', text)
    #menghilangkan tanda baca
    text = text.translate(str.maketrans("","",string.punctuation))
    # menghilangkan whitespace berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    return text

data_tweet['filtered'] = data_tweet['full_text'].apply(filtering_text)
print(data_tweet['filtered'].head())

0    lowongan kerja bumn lpdp kemenkeu terbaru dese...
1    ku wes cukup absurd tulong ku pengen luwih ter...
2                                          lpdp apa ya
3        lowongan rekrutmen lpdp kemenkeu info lengkap
4    hai lpdprens terima kasih atas sharingnya ya s...
Name: filtered, dtype: object


In [ ]:
# ==== load kamus sekali ====
with open('/content/drive/MyDrive/Int Conference/Slang/kamus.txt') as kamus:
    word = kamus.readlines()
    list_stopword = [line.replace('\n',"") for line in word]

dictionary = ArrayDictionary(list_stopword)
stopword = StopWordRemover(dictionary)

# ==== buat stemmer sekali ====
factory_stemmer = StemmerFactory()
stemmer = factory_stemmer.create_stemmer()

# ==== fungsi ====
def stop_stem(text):
    text = stopword.remove(text)
    text = stemmer.stem(text)
    return text

data_tweet['cleaned'] = data_tweet['filtered'].apply(stop_stem)

In [ ]:
def word_tokenize_wrapper(text):
    return word_tokenize(text)

data_tweet['tweet_tokens'] = data_tweet['cleaned'].apply(word_tokenize_wrapper)

print(data_tweet['tweet_tokens'].head())

0    [lowong, kerja, bumn, lpdp, kemenkeu, baru, de...
1    [ku, wes, cukup, absurd, tulong, ku, ken, luwi...
2                                          [lpdp, apa]
3    [lowong, rekrutmen, lpdp, kemenkeu, info, leng...
4    [hai, lpdprens, terima, kasih, atas, sharingny...
Name: tweet_tokens, dtype: object


In [ ]:
kamus_normalisasi = pd.read_csv('/content/drive/MyDrive/Int Conference/Slang/slang.csv')

kata_normalisasi_dict = {}

for index, row in kamus_normalisasi.iterrows():
    if row[0] not in kata_normalisasi_dict:
        kata_normalisasi_dict[row[0]] = row[1]

def normalisasi_kata(document):
    return [kata_normalisasi_dict[term] if term in kata_normalisasi_dict else term for term in document]

data_tweet['normalisasi'] = data_tweet['tweet_tokens'].apply(normalisasi_kata)

data_tweet['normalisasi'].head(10)

/tmp/ipykernel_1383/3904099933.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if row[0] not in kata_normalisasi_dict:
/tmp/ipykernel_1383/3904099933.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  kata_normalisasi_dict[row[0]] = row[1]


,normalisasi
0,"[lowong, kerja, bumn, lpdp, kemenkeu, baru, de..."
1,"[ku, wes, cukup, absurd, tulong, ku, ken, luwi..."
2,"[lpdp, apa]"
3,"[lowong, rekrutmen, lpdp, kemenkeu, info, leng..."
4,"[hai, lpdprens, terima, kasih, atas, sharingny..."
5,[]
6,"[bismillah, coba, lpdp, ahhh]"
7,"[ri, hardwork, dengan, lazyness]"
8,"[dapat, kgsp, kalau, tidak, lpdp, amin]"
9,[]


In [ ]:
data_tweet.to_csv('/content/drive/MyDrive/Int Conference/Slang/preprocess.csv', index=False)